# 한국어 입문용 TraceWhisperer 예제
이 노트북은 **ChipWhisperer Husky + CW308_STM32F3 + simpleserial-trace 펌웨어 + SWO 모드**를 기준으로 작성한 **TraceWhisperer 입문 예제**입니다.

목표는 아래 3가지를 한 번에 이해하는 것입니다.

1. **TraceWhisperer를 최소 설정으로 켠다.**
2. **펌웨어 트리거와 함께 trace event timestamp를 얻는다.**
3. **얻은 timestamp를 전력 파형 위에 겹쳐서 본다.**

## 이 예제의 대상
- ChipWhisperer 기본 캡처는 해본 적이 있음
- Husky와 STM32F3 타깃 연결은 할 수 있음
- TraceWhisperer는 처음이거나, 공식 예제를 그대로 따라만 본 적이 있음

## 준비물
- ChipWhisperer Husky
- CW308 + STM32F3 타깃
- `simpleserial-trace` 펌웨어
- Jupyter 환경에서 ChipWhisperer 예제 실행 가능 상태

## 먼저 알아둘 점
- 아래 주소 `0x080018c4`, `0x0800188c` 는 **예시 펌웨어 빌드 기준**입니다.
- 펌웨어를 다시 빌드했거나 버전이 다르면 함수 주소가 바뀔 수 있습니다.
- 그래서 뒤쪽에 **ELF에서 함수 주소를 찾는 선택 예제**도 넣었습니다.

## 1. 실습용 설정값
가장 먼저, 실습에서 바꾸기 쉬운 값들을 한 곳에 모아 둡니다.

In [ ]:
PLATFORM = "CW308_STM32F3"
SCOPETYPE = "OPENADC"

# ChipWhisperer Jupyter 예제 디렉터리 기준 상대 경로입니다.
# 노트북 위치가 다르면 이 경로를 수정하세요.
SETUP_NOTEBOOK = "../base/Setup_Generic.ipynb"

# simpleserial-trace 펌웨어 경로
FW_HEX_PATH = "../../../firmware/mcu/simpleserial-trace/simpleserial-trace-CW308_STM32F3.hex"
FW_ELF_PATH = "../../../firmware/mcu/simpleserial-trace/simpleserial-trace-CW308_STM32F3.elf"

# 이미 올바른 펌웨어가 올라가 있으면 False로 두고 진행하세요.
DO_PROGRAM_TARGET = True

# Husky 권장 시작값
ADC_SAMPLES = 31000
GAIN_DB = 12

# TraceWhisperer 기본 설정
TRACE_CLOCK_SOURCE = "target_clock"
TRACE_INTERFACE = "SWO"

# SWO 분주 관련 설정
SWO_TRIGGER_FREQ_MUL = 8
SWO_ACPR = 0

# 예제 펌웨어 기준 함수 시작 주소
# addr0: SubBytes
# addr1: AddRoundKey
TRACE_MATCH_ADDR0 = 0x080018C4
TRACE_MATCH_ADDR1 = 0x0800188C

## 2. 기본 import와 보드 연결
이 단계에서는 ChipWhisperer 기본 환경을 올리고, Husky의 trace 블록을 활성화합니다.

실행 후 확인할 것:
- `scope.trace.enabled = True`
- `scope.trace.target = target`
- `scope.gain.db`, `scope.adc.samples` 값이 정상 반영되었는지

In [ ]:
import time
import re
import subprocess
import numpy as np
import chipwhisperer as cw

from bokeh.plotting import figure, show
from bokeh.io import output_notebook
from bokeh.models import Span

output_notebook()

# ChipWhisperer의 기본 setup notebook 실행
%run "../base/Setup_Generic.ipynb"

# Husky의 trace 블록과 타깃 연결
scope.trace.target = target
scope.trace.enabled = True

# 전력 파형 캡처 설정
scope.adc.samples = ADC_SAMPLES
scope.gain.db = GAIN_DB

print("adc.samples   =", scope.adc.samples)
print("gain.db       =", scope.gain.db)
print("trace.enabled =", scope.trace.enabled)

## 3. 리셋 및 상태 확인 함수
TraceWhisperer 실습에서는 **리셋 후 다시 시도**가 매우 중요합니다.

특히 아래와 같은 경우에는 리셋이 자주 필요합니다.
- 펌웨어 프로그래밍 직후
- SWO/JTAG 전환 직후
- `capture failed`
- `fifo_empty() == True`
- `get_fw_buildtime()` 이 비어 있거나 응답이 이상함

In [ ]:
EXPECTED_FW_HINT = "ChipWhisperer simpleserial-trace"

def reset_target_hard(scope):
    # Husky + CW308 조합에서 많이 쓰는 NRST 토글 리셋
    scope.io.nrst = "low"
    time.sleep(0.05)
    scope.io.nrst = "high"
    time.sleep(0.05)

def get_fw_buildtime_safe(scope):
    # 펌웨어 응답 문자열을 안전하게 읽기
    try:
        return scope.trace.get_fw_buildtime()
    except Exception as e:
        return f"<fw_buildtime read failed: {e}>"

def print_basic_status(scope):
    print("FW buildtime :", get_fw_buildtime_safe(scope))
    try:
        print("Husky FPGA   :", scope.fpga_buildtime)
    except Exception as e:
        print("Husky FPGA   : <unavailable>", e)

reset_target_hard(scope)
print_basic_status(scope)

## 4. 필요 시 타깃 펌웨어 프로그래밍
이미 `simpleserial-trace` 펌웨어가 올라가 있으면 이 셀은 건너뛰어도 됩니다.

### 언제 이 셀을 실행하면 좋은가?
- `get_fw_buildtime()` 결과에 `ChipWhisperer simpleserial-trace` 가 보이지 않을 때
- 캡처가 반복적으로 실패할 때
- 실습 환경을 처음 세팅할 때

In [ ]:
if DO_PROGRAM_TARGET:
    print("Programming target firmware...")
    prog = cw.programmers.STM32FProgrammer
    cw.program_target(scope, prog, FW_HEX_PATH)
    reset_target_hard(scope)
    time.sleep(0.1)

print_basic_status(scope)

fw_info = get_fw_buildtime_safe(scope)
assert EXPECTED_FW_HINT in fw_info, (
    "올바른 simpleserial-trace 펌웨어가 아닌 것 같습니다. "
    "FW_HEX_PATH를 확인하거나 DO_PROGRAM_TARGET=True로 다시 시도하세요."
)

## 5. SWO 모드로 전환
TraceWhisperer는 병렬 trace와 SWO를 모두 다룰 수 있지만, 입문자에게는 **SWO 모드**가 이해하기 쉽습니다.

이 단계의 핵심은 4가지입니다.
1. Front-end clock source 선택
2. SWO 모드로 전환
3. JTAG → SWD 전환
4. SWO clock lock 확인

실행 후 가장 중요하게 볼 값:
- `scope.trace.clock.fe_clock_alive`
- `scope.trace.clock.swo_clock_locked`

In [ ]:
assert TRACE_INTERFACE == "SWO", "이 입문 예제는 SWO 기준으로 작성되었습니다."

scope.trace.clock.fe_clock_src = TRACE_CLOCK_SOURCE
assert scope.trace.clock.fe_clock_alive, (
    "선택한 front-end clock이 살아 있지 않습니다. "
    "target_clock 연결 상태와 타깃 동작 여부를 확인하세요."
)

scope.trace.trace_mode = "SWO"
scope.trace.jtag_to_swd()

# 공식 Husky 예제의 기본 안정값
scope.trace.clock.swo_clock_freq = scope.clock.clkgen_freq * SWO_TRIGGER_FREQ_MUL
scope.trace.target_registers.TPI_ACPR = SWO_ACPR
scope.trace.swo_div = SWO_TRIGGER_FREQ_MUL * (SWO_ACPR + 1)

assert scope.trace.clock.swo_clock_locked, (
    "SWO clock lock 실패. "
    "케이블 연결, 클럭 설정, 타깃 리셋을 다시 확인하세요."
)

# Husky 버전에 따라 userio 접근 방식이 조금 다를 수 있어 둘 다 시도
swo_line_ok = None
try:
    swo_line_ok = bool(scope.userio.pins[2].status)
except Exception:
    try:
        swo_line_ok = bool(scope.userio.status & 0x4)
    except Exception:
        swo_line_ok = None

print("fe_clock_alive   =", scope.trace.clock.fe_clock_alive)
print("swo_clock_locked =", scope.trace.clock.swo_clock_locked)
print("swo_line_ok      =", swo_line_ok)

## 6. 불필요한 sync frame 끄기
처음 bring-up 단계에서는 sync frame이 도움이 되지만, 입문 실습에서는 **관심 있는 trace event를 더 깔끔하게 보기 위해 끄는 편이 좋습니다.**

In [ ]:
scope.trace.target_registers.DWT_CTRL = 0x40000021
print("DWT_CTRL =", hex(scope.trace.target_registers.DWT_CTRL))

## 7. 어떤 이벤트를 잡을지 정의
우리는 raw packet 전체를 저장하지 않고,  
**특정 패턴이 보일 때의 timestamp만 저장**하는 모드로 시작합니다.

### 이번 예제의 의미
- `raw = False`
- `set_pattern_match(0, [3, 8, 32])`
- `rules_enabled = [0]`

즉, 특정 trace 패턴이 보일 때마다 그 시점을 기록합니다.
초보자는 이 모드부터 시작하는 것이 가장 이해하기 쉽습니다.

In [ ]:
scope.trace.capture.raw = False

# STM32F3 + simpleserial-trace 예제에서 많이 사용하는 기본 패턴
scope.trace.set_pattern_match(0, [3, 8, 32])

# rule 0 활성화
scope.trace.capture.rules_enabled = [0]

print("capture.raw   =", scope.trace.capture.raw)
print("rules_enabled =", scope.trace.capture.rules_enabled)

## 8. 캡처 시작/종료 조건 설정
이번 예제에서는 **펌웨어가 TIO4로 생성하는 trigger 구간 동안만** trace를 잡습니다.

이 방식이 입문자에게 좋은 이유:
- 전력 파형 구간과 trace 구간을 맞춰 보기 쉬움
- 캡처 시간이 짧아져서 실패 원인 분석이 쉬움

In [ ]:
scope.trace.capture.trigger_source = "firmware trigger"
scope.trace.capture.mode = "while_trig"

print("trigger_source =", scope.trace.capture.trigger_source)
print("capture.mode   =", scope.trace.capture.mode)

## 9. 함수 주소를 매칭 대상으로 설정
TraceWhisperer를 쓰는 가장 직관적인 방법 중 하나는  
**특정 함수 시작 주소에 대한 trace event를 잡는 것**입니다.

이번 예제에서는 아래 두 함수를 대상으로 둡니다.
- `SubBytes`
- `AddRoundKey`

### 매우 중요한 주의사항
아래 주소는 예제 펌웨어 빌드 기준입니다.
펌웨어가 다르면 주소도 바뀔 수 있습니다.

In [ ]:
scope.trace.set_isync_matches(
    addr0=TRACE_MATCH_ADDR0,
    addr1=TRACE_MATCH_ADDR1,
    match="both"
)

# 입문 예제에서는 periodic PC sampling을 끕니다.
# 이유: 결과 해석을 단순하게 유지하기 위해
scope.trace.set_periodic_pc_sampling(enable=0)

print("addr0 =", hex(TRACE_MATCH_ADDR0))
print("addr1 =", hex(TRACE_MATCH_ADDR1))

## 10. 1회 캡처 함수 만들기
이제 한 번의 캡처를 반복 가능하게 함수로 정리합니다.

이 함수가 하는 일:
1. 필요 시 리셋
2. TraceWhisperer arm
3. 전력 파형 캡처
4. trace FIFO 읽기
5. timestamp 추출

In [ ]:
def capture_once(scope, target, text=None, key=None, reset_before=False, verbose=True):
    if reset_before:
        reset_target_hard(scope)
        time.sleep(0.1)

    if text is None:
        text = bytearray(16)
    if key is None:
        key = bytearray(16)

    # TraceWhisperer arm
    scope.trace.arm_trace()

    # 전력 + 트리거 + TraceWhisperer 동시 캡처
    powertrace = cw.capture_trace(scope, target, text, key)
    assert powertrace is not None, (
        "capture_trace() 실패. 타깃 응답, 케이블 연결, 리셋 상태를 확인하세요."
    )

    assert not scope.trace.fifo_empty(), (
        "TraceWhisperer FIFO가 비어 있습니다. "
        "trace arm 타이밍, SWO 설정, 함수 주소, trigger 구간을 확인하세요."
    )

    raw = scope.trace.read_capture_data()
    times = scope.trace.get_rule_match_times(raw, rawtimes=False, verbose=verbose)

    return powertrace, raw, times

## 11. 실제 1회 캡처
정상이라면 `times` 안에 적어도 1개 이상의 이벤트가 들어옵니다.

### 기대하는 결과
- `powertrace.wave` 는 전력 파형
- `raw` 는 TraceWhisperer가 읽은 내부 raw 데이터
- `times` 는 패턴이 매칭된 시간 정보

In [ ]:
powertrace, raw, times = capture_once(
    scope=scope,
    target=target,
    text=bytearray(16),
    key=bytearray(16),
    reset_before=False,
    verbose=True
)

print("number of trace events =", len(times))
print("times =", times)
print("matched_pattern_data   =", scope.trace.capture.matched_pattern_data)
print("matched_pattern_counts =", scope.trace.capture.matched_pattern_counts)

## 12. 전력 파형 위에 trace event 표시
입문자가 가장 좋아하는 구간입니다.

전력 파형만 보면 "어디가 SubBytes인지" 감이 잘 오지 않지만,  
TraceWhisperer event를 세로선으로 표시하면  
**함수 경계나 관심 코드 위치를 훨씬 직관적으로 볼 수 있습니다.**

In [ ]:
def get_adc_multiplier(scope):
    # Husky에서는 adc_mul 사용
    if getattr(scope, "_is_husky", False):
        return scope.clock.adc_mul

    # 다른 보드용 일반 fallback
    adc_src = getattr(scope.clock, "adc_src", None)
    if adc_src in ("clkgen_x4", "extclk_x4"):
        return 4
    return 1

multiplier = get_adc_multiplier(scope)

p = figure(
    width=1300,
    height=350,
    title="Power trace + TraceWhisperer event timestamps"
)

xrange = list(range(len(powertrace.wave)))
p.line(xrange, powertrace.wave)

for idx, t in enumerate(times):
    p.add_layout(
        Span(
            location=t[0] * multiplier,
            dimension="height",
            line_width=2
        )
    )

show(p)

## 13. 여러 번 반복 캡처해 보기
처음에는 1회 캡처가 더 이해하기 쉽지만,  
실습이 안정화되면 여러 번 반복해서 **event 위치가 얼마나 안정적인지** 보는 것이 좋습니다.

In [ ]:
all_event_times = []

for i in range(5):
    powertrace_i, raw_i, times_i = capture_once(
        scope=scope,
        target=target,
        text=bytearray(16),
        key=bytearray(16),
        reset_before=False,
        verbose=False
    )
    all_event_times.append(times_i)

for i, times_i in enumerate(all_event_times):
    print(f"[capture {i}] {times_i}")

## 14. 함수 주소가 안 맞는 경우: ELF에서 주소 찾기
펌웨어를 다시 빌드했으면 함수 시작 주소가 달라질 수 있습니다.

이 셀은 `arm-none-eabi-objdump` 를 사용해서 ELF에서 함수 시작 주소를 찾습니다.

### 자주 쓰는 상황
- `times` 가 비어 있음
- 패턴은 잡히는데 원하는 함수가 아닌 것 같음
- 다른 버전의 firmware를 직접 빌드함

In [ ]:
def find_function_addresses_from_elf(elf_path, function_names):
    result = subprocess.run(
        ["arm-none-eabi-objdump", "-d", elf_path],
        capture_output=True,
        text=True,
        check=True
    )

    head_regex = re.compile(r"^([0-9a-fA-F]{8})\s<([^>]+)>:$")
    found = {}

    for line in result.stdout.splitlines():
        m = head_regex.match(line.strip())
        if m and m.group(2) in function_names:
            found[m.group(2)] = int(m.group(1), 16)

    return found

try:
    found = find_function_addresses_from_elf(
        FW_ELF_PATH,
        ["SubBytes", "AddRoundKey", "ShiftRows", "KeyExpansion"]
    )
    print(found)
except FileNotFoundError:
    print("arm-none-eabi-objdump 가 없습니다. ChipWhisperer 빌드 환경에서 다시 시도하세요.")
except subprocess.CalledProcessError as e:
    print("objdump 실행 실패:", e)

## 15. 선택 실습: 찾은 주소로 다시 매칭하기
위에서 찾은 주소를 실제 캡처 설정에 반영하는 셀입니다.

In [ ]:
# 예:
# found = {'SubBytes': 0x080018c4, 'AddRoundKey': 0x0800188c}

if "found" in globals():
    if "SubBytes" in found and "AddRoundKey" in found:
        scope.trace.set_isync_matches(
            addr0=found["SubBytes"],
            addr1=found["AddRoundKey"],
            match="both"
        )
        print("Updated addresses from ELF:")
        print("  SubBytes    =", hex(found["SubBytes"]))
        print("  AddRoundKey =", hex(found["AddRoundKey"]))
    else:
        print("필요한 함수 주소를 모두 찾지 못했습니다.")
else:
    print("먼저 바로 위 셀을 실행하세요.")

## 16. 선택 실습: TraceWhisperer Explorer 띄우기
공식 탐색 노트북처럼 함수 단위로 trace를 더 직관적으로 보고 싶다면 `TraceWhispererExplorer` 를 사용할 수 있습니다.

이 부분은 입문 필수는 아니고,  
**"trace event가 실제 코드 어느 부분과 연결되는지 더 시각적으로 보고 싶다"** 는 경우에 추천합니다.

In [ ]:
def build_func_list_from_elf(elf_path, useful_functions):
    result = subprocess.run(
        ["arm-none-eabi-objdump", "-d", elf_path],
        capture_output=True,
        text=True,
        check=True
    )

    head_regex = re.compile(r"(\w{8})\s<(\w+)>:$")
    diss_regex = re.compile(r"\s(\w+):")

    dumplines = result.stdout.split("\n")
    funcs = []

    i = 0
    while i < len(dumplines):
        line = dumplines[i]
        match = head_regex.search(line)
        if match:
            start = match.group(1)
            name = match.group(2)
            if name in useful_functions:
                j = 1
                diss = ""
                last_addr = start
                while i + j < len(dumplines):
                    nextline = dumplines[i + j]
                    m2 = diss_regex.search(nextline)
                    if m2:
                        diss += nextline + "\n"
                        last_addr = m2.group(1)
                        j += 1
                    else:
                        break
                funcs.append([int(start, 16), int(last_addr, 16), diss, name])
                i += j
                continue
        i += 1

    return funcs

useful_functions = [
    "KeyExpansion",
    "AddRoundKey",
    "SubBytes",
    "ShiftRows",
    "xtime",
    "Cipher",
    "BlockCopy",
    "AES128_ECB_indp_crypto",
]

try:
    funcs = build_func_list_from_elf(FW_ELF_PATH, useful_functions)
    print("functions parsed =", len(funcs))
except FileNotFoundError:
    print("arm-none-eabi-objdump 가 없습니다.")
except subprocess.CalledProcessError as e:
    print("objdump 실행 실패:", e)

In [ ]:
try:
    from chipwhisperer.common.utils.tracewhisperer_explorer import TraceWhispererExplorer

    if "funcs" in globals() and len(funcs) > 0:
        te = TraceWhispererExplorer(scope, target, funcs, width=1600, height=400)
        print("TraceWhispererExplorer 생성 완료")
    else:
        print("funcs 가 비어 있습니다. 바로 위 셀을 먼저 실행하세요.")
except ImportError as e:
    print("TraceWhispererExplorer import 실패:", e)

## 17. 초보자용 문제 해결 체크리스트

### 1) `fe_clock_alive == False`
- target clock이 실제로 살아 있는지 확인
- 타깃 전원이 정상인지 확인
- Setup notebook 경로가 맞는지 확인

### 2) `swo_clock_locked == False`
- SWO 배선 또는 설정 재확인
- 타깃 리셋 후 다시 시도
- `SWO_TRIGGER_FREQ_MUL`, `SWO_ACPR` 를 공식 예제 기본값으로 유지

### 3) `capture_trace() failed`
- 타깃이 응답 가능한 상태인지 확인
- `reset_target_hard(scope)` 후 재시도
- 펌웨어가 `simpleserial-trace` 인지 확인

### 4) `fifo_empty() == True`
- `scope.trace.arm_trace()` 를 캡처 직전에 실행했는지 확인
- 함수 주소가 현재 ELF와 맞는지 확인
- trigger 구간이 너무 짧지 않은지 확인

### 5) 이벤트 수는 나오는데 위치가 이상함
- 함수 주소 재확인
- 다른 firmware 빌드를 사용 중인지 확인
- periodic PC sampling이 꺼져 있는지 확인

## 18. 이 노트북의 학습 포인트 정리
이 예제에서 꼭 가져가야 할 핵심은 아래입니다.

1. **TraceWhisperer는 전력 파형을 대체하는 도구가 아니라, 전력 파형 해석을 보조하는 시간 기준 도구**로 매우 강력합니다.
2. 초보자는 처음부터 raw trace packet을 다루기보다, **pattern match + timestamp 방식**으로 시작하는 것이 훨씬 쉽습니다.
3. 실무/연구에서는 결국 **"관심 함수 주소를 정확히 매칭하는 것"** 이 품질을 크게 좌우합니다.
4. 같은 펌웨어라도 다시 빌드하면 주소가 달라질 수 있으므로, **ELF에서 주소를 다시 찾는 습관**이 매우 중요합니다.